# Consulting Project 
#### Task

The whole world seems to be hearing about your new amazing abilities to analyze big data and build useful systems for them! You've just taken up a new contract with a new online food delivery company. This company is trying to differentiate itself by recommending new meals to customers based off of other customers likings.

Can you build them a recommendation system?

Your final result should be in the form of a function that can take in a Spark DataFrame of a single customer's ratings for various meals and output their top 3 suggested meals.

#### Import required libraries for Spark processing and sklearn recommender modeling.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error

#### Define a mapping from meal IDs to human-readable meal names for final recommendations.

In [0]:
mealmap = { 2. : "Chicken Curry",   
           3. : "Spicy Chicken Nuggest",   
           5. : "Hamburger",   
           9. : "Taco Surprise",  
           11. : "Meatloaf",  
           12. : "Ceaser Salad",  
           15. : "BBQ Ribs",  
           17. : "Sushi Plate",  
           19. : "Cheesesteak Sandwhich",  
           21. : "Lasagna",  
           23. : "Orange Chicken",
           26. : "Spicy Beef Plate",  
           27. : "Salmon with Mashed Potatoes",  
           28. : "Penne Tomatoe Pasta",  
           29. : "Pork Sliders",  
           30. : "Vietnamese Sandwich",  
           31. : "Chicken Wrap",  
           np.nan: "Cowboy Burger",   
           4. : "Pretzels and Cheese Plate",   
           6. : "Spicy Pork Sliders",  
           13. : "Mandarin Chicken PLate",  
           14. : "Kung Pao Chicken",
           16. : "Fried Rice Plate",  
           8. : "Chicken Chow Mein",  
           10. : "Roasted Eggplant ",  
           18. : "Pepperoni Pizza",  
           22. : "Pulled Pork Plate",   
           0. : "Cheese Pizza",   
           1. : "Burrito",   
           7. : "Nachos",  
           24. : "Chili",  
           20. : "Southwest Salad",  
           25.: "Roast Beef Sandwich"}

#### Start Spark session and load meal ratings data into a Spark DataFrame.

In [0]:
spark = SparkSession.builder.appName('recconsulting').getOrCreate()
ratings_spark_df = spark.read.csv('data/meal_info.csv', inferSchema=True, header=True)
ratings_spark_df = ratings_spark_df.select('userId', 'mealskew', 'rating')

In [0]:
ratings_spark_df.show(5)

+------+--------+------+
|userId|mealskew|rating|
+------+--------+------+
|     0|     2.0|   3.0|
|     0|     3.0|   1.0|
|     0|     5.0|   2.0|
|     0|     9.0|   4.0|
|     0|    11.0|   1.0|
+------+--------+------+
only showing top 5 rows


#### Split ratings into train and test sets.

In [0]:
training_spark_df, test_spark_df = ratings_spark_df.randomSplit([0.8, 0.2], seed=42)

#### Convert train/test to pandas, build user-item matrix, and fit NMF latent-factor model.

In [0]:
train_pdf = training_spark_df.toPandas()
test_pdf = test_spark_df.toPandas()

train_matrix = train_pdf.pivot_table(index='userId', columns='mealskew', values='rating', fill_value=0.0)
train_users = train_matrix.index.to_numpy()
train_items = train_matrix.columns.to_numpy()

nmf_model = NMF(n_components=8, init='nndsvda', random_state=42, max_iter=400)
user_factors = nmf_model.fit_transform(train_matrix.values)
item_factors = nmf_model.components_
reconstructed_train = user_factors @ item_factors

#### Build helper mappings and a rating prediction function from reconstructed latent factors.

In [0]:
user_to_pos = {user_id: idx for idx, user_id in enumerate(train_users)}
item_to_pos = {item_id: idx for idx, item_id in enumerate(train_items)}

def predict_rating(user_id, meal_id):
    if user_id not in user_to_pos or meal_id not in item_to_pos:
        return np.nan
    u_idx = user_to_pos[user_id]
    i_idx = item_to_pos[meal_id]
    return float(reconstructed_train[u_idx, i_idx])

#### Evaluate RMSE on test data and define a top-3 recommendation function for a single customer.

In [0]:
test_pdf = test_pdf.copy()
test_pdf['prediction'] = test_pdf.apply(lambda row: predict_rating(row['userId'], row['mealskew']), axis=1)

eval_pdf = test_pdf.dropna(subset=['prediction'])
rmse = mean_squared_error(eval_pdf['rating'], eval_pdf['prediction'], squared=False)
print('Root-mean-square error = ' + str(rmse))

def top_3_recommendations_for_user(customer_spark_df):
    customer_pdf = customer_spark_df.select('userId', 'mealskew', 'rating').toPandas()
    user_id = customer_pdf['userId'].iloc[0]
    known_meals = set(customer_pdf['mealskew'])

    if user_id in user_to_pos:
        user_scores = reconstructed_train[user_to_pos[user_id]]
    else:
        user_scores = reconstructed_train.mean(axis=0)

    scored_meals = []
    for meal_id, pos in item_to_pos.items():
        if meal_id not in known_meals:
            scored_meals.append((meal_id, float(user_scores[pos])))

    top_meals = sorted(scored_meals, key=lambda x: x[1], reverse=True)[:3]
    return [(mealmap.get(meal_id, meal_id), score) for meal_id, score in top_meals]

sample_customer = ratings_spark_df.filter('userId = 11').limit(10)
print(top_3_recommendations_for_user(sample_customer))

Root-mean-square error = 1.737689482945869
[('Salmon with Mashed Potatoes', 5.489765025847967), ('Vietnamese Sandwich', 5.07342603337595), ('Orange Chicken', 3.8793722977399403)]
